# 08 · Multi-resolution sensitivity (250 m – ~1 km) — Turkish Thrace VHI  *(v2)*

**Purpose.** Test whether the two headline patterns — the SPEI-3 monitoring optimum and the
rainfed-plateau vs irrigated-İpsala coupling contrast — survive when the 30 m Landsat VHI is
aggregated to coarse (MODIS/AVHRR-like) resolution. Turns the manuscript's §4.3 argument into a
demonstrated result.

**Design.** The same NDVI/LST are aggregated to 30 m (control), 250 m, 500 m and 900 m (~1 km), in three arms:
- **A — native masked (30 m):** CORINE-211 cropland-masked district mean = the paper's result.
- **B — coarse masked:** cropland-masked, *then* aggregated to R m (mask kept → isolates pure resolution loss).
- **C — coarse unmasked:** aggregated to R m with *no* cropland mask (a real coarse sensor that cannot resolve
  cropland → mixes rainfed, irrigated and non-crop land).

*v2 fix:* each arm × resolution is extracted **separately** (so one failing combination can't null the
whole run), columns are tagged Python-side, and the coarsest step is 900 m to stay under the
`reduceResolution` aggregation cap.

Run top to bottom. The slow cell is §5. Export the CSV at the end and send it back.


## 0 · Setup

In [ ]:
!pip install -q geemap earthengine-api
import ee, geemap, numpy as np, pandas as pd
ee.Authenticate()
ee.Initialize(project='vhi-landsat')   # <-- your Cloud project id
print('EE:', ee.String('OK').getInfo())

## 1 · Parameters

In [ ]:
# Resolution-test window: default L8/L9 era (single-generation sensor, avoids the 2013 thermal
# handover — appropriate for a resolution SENSITIVITY test). Set 1985 for the full record (slower).
START_YEAR = 2013
END_YEAR   = 2024

R_LIST     = [250, 500, 900]     # coarse targets (m); 900 ~ 1 km, kept < the reduceResolution 1024-native-px cap
SENSORS    = ['L8','L9'] if START_YEAR >= 2013 else ['L5','L7','L8','L9']
HARMONIZE  = True                # Roy 2016 (only used if L5/L7 included)
CLOUD_BITS = True

# Explicit, validated yield-district -> (GAUL province, GAUL ADM2). Provincial capitals are 'Merkez'
# in GAUL 2015, disambiguated by province. Six rainfed plateau districts + İpsala (contrast).
DISTRICT_MAP = {
    'Edirne':     ('Edirne',     'Merkez'),
    'Uzunkopru':  ('Edirne',     'Uzunkopru'),
    'Ipsala':     ('Edirne',     'Ipsala'),
    'Luleburgaz': ('Kirklareli', 'Luleburgaz'),
    'Kirklareli': ('Kirklareli', 'Merkez'),
    'Tekirdag':   ('Tekirdag',   'Merkez'),
    'Corlu':      ('Tekirdag',   'Corlu'),
}
PLATEAU6 = ['Corlu','Edirne','Kirklareli','Luleburgaz','Tekirdag','Uzunkopru']  # İpsala = contrast

## 2 · District polygons (GAUL) + ROI

In [ ]:
gaul = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM0_NAME','Turkey'))
def make_district(key, prov, adm2):
    f = gaul.filter(ee.Filter.And(ee.Filter.eq('ADM1_NAME', prov),
                                  ee.Filter.eq('ADM2_NAME', adm2))).first()
    return ee.Feature(ee.Feature(f).geometry(), {'name': key})
districts = ee.FeatureCollection([make_district(k,p,a) for k,(p,a) in DISTRICT_MAP.items()])
print('District polygons built:', districts.size().getInfo())
print('Districts:', sorted(districts.aggregate_array('name').getInfo()))
roi = districts.geometry().bounds()

## 3 · CORINE-211 rainfed-cropland mask

In [ ]:
corine   = ee.Image('COPERNICUS/CORINE/V20/100m/2018').select('landcover')
cropland = corine.eq(211).selfMask().rename('cropland')
cnt = cropland.reduceRegion(ee.Reducer.count(), roi, 100, maxPixels=1e10).getInfo()
n211 = list(cnt.values())[0] if cnt else 0
print('CORINE 211 pixels in ROI:', n211, '| OK' if n211 else '*** empty — use ESA WorldCover fallback ***')

## 4 · Landsat C2 L2 collection (NDVI + surface-temperature LST)

In [ ]:
SR_SCALE,SR_OFF = 0.0000275,-0.2; ST_SCALE,ST_OFF = 0.00341802,149.0
SENSOR_DATES={'L5':('1984-03-01','2012-05-05'),'L7':('1999-05-28','2022-04-06'),
              'L8':('2013-03-18','2025-12-31'),'L9':('2021-10-31','2025-12-31')}
CFG={'L5':('LANDSAT/LT05/C02/T1_L2',['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7','ST_B6','QA_PIXEL']),
     'L7':('LANDSAT/LE07/C02/T1_L2',['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7','ST_B6','QA_PIXEL']),
     'L8':('LANDSAT/LC08/C02/T1_L2',['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7','ST_B10','QA_PIXEL']),
     'L9':('LANDSAT/LC09/C02/T1_L2',['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7','ST_B10','QA_PIXEL'])}
COMMON=['blue','green','red','nir','swir1','swir2','thermal','QA_PIXEL']; OPT=['blue','green','red','nir','swir1','swir2']
ROY_S={'blue':0.8474,'green':0.8483,'red':0.9047,'nir':0.8462,'swir1':0.8937,'swir2':0.9071}
ROY_I={'blue':0.0003,'green':0.0088,'red':0.0061,'nir':0.0412,'swir1':0.0254,'swir2':0.0172}

def mask_qa(img):
    qa=img.select('QA_PIXEL')
    bad=(qa.bitwiseAnd(1<<1).neq(0).Or(qa.bitwiseAnd(1<<2).neq(0)).Or(qa.bitwiseAnd(1<<3).neq(0))
         .Or(qa.bitwiseAnd(1<<4).neq(0)).Or(qa.bitwiseAnd(1<<5).neq(0)))
    return img.updateMask(bad.Not())

def prep(sensor):
    coll_id,bands=CFG[sensor]; s0,s1=SENSOR_DATES[sensor]
    lo=f'{max(int(s0[:4]),START_YEAR)}-01-01'; hi=f'{min(int(s1[:4]),END_YEAR)}-12-31'
    coll=ee.ImageCollection(coll_id).filterBounds(roi).filterDate(lo,hi)
    if CLOUD_BITS: coll=coll.map(mask_qa)
    def _p(img):
        img=img.select(bands,COMMON); opt=img.select(OPT).multiply(SR_SCALE).add(SR_OFF)
        if HARMONIZE and sensor in ('L5','L7'):
            opt=opt.multiply(ee.Image([ROY_S[b] for b in OPT])).add(ee.Image([ROY_I[b] for b in OPT]))
        lst=img.select('thermal').multiply(ST_SCALE).add(ST_OFF).subtract(273.15).rename('LST')
        ndvi=opt.normalizedDifference(['nir','red']).rename('NDVI')
        return ndvi.addBands(lst).copyProperties(img,['system:time_start'])
    return coll.map(_p)

landsat=prep(SENSORS[0])
for s in SENSORS[1:]: landsat=landsat.merge(prep(s))
landsat=landsat.select(['NDVI','LST'])
print('Collection built (lazy) for', SENSORS, f'{START_YEAR}-{END_YEAR}.')

## 5 · Multi-resolution district extraction  *(slow cell)*

Each arm × resolution is extracted independently (nb01-style per-year pattern). Coarse arms aggregate
30 m NDVI/LST to R m with `reduceResolution(mean)` before the district reduction.

In [ ]:
def prep_image(img, R, masked, do_coarsen):
    im = img.updateMask(cropland) if masked else img
    if do_coarsen:
        im = (im.reduceResolution(ee.Reducer.mean(), maxPixels=1024)
                .reproject(crs=img.select('NDVI').projection().crs(), scale=R))
    return im

def extract_arm_year(R, masked, do_coarsen, year):
    coll = landsat.filterDate(f'{year}-01-01', f'{year}-12-31')
    def samp(img):
        t = img.get('system:time_start')
        im = prep_image(img, R, masked, do_coarsen)
        return (im.reduceRegions(collection=districts, reducer=ee.Reducer.mean(),
                                 scale=max(R,30), tileScale=4)
                  .map(lambda f: f.set('millis', t)))
    return geemap.ee_to_df(coll.map(samp).flatten())

# (arm_name, R, masked, do_coarsen)
ARMS = [('A_native_masked', 30, True, False)]
for R in R_LIST:
    ARMS.append(('B_coarse_masked',   R, True,  True))
    ARMS.append(('C_coarse_unmasked', R, False, True))

frames=[]
for arm,R,masked,do_c in ARMS:
    got=0
    for y in range(START_YEAR, END_YEAR+1):
        try:
            d = extract_arm_year(R, masked, do_c, y)
        except Exception as e:
            print(f'  {arm} R={R} {y}: skipped ({str(e)[:60]})'); continue
        if len(d) and 'millis' in d.columns:
            d['arm']=arm; d['res_m']=R; frames.append(d); got+=len(d)
    print(f'{arm:18s} R={R:<4} rows={got}')

if not frames:
    raise RuntimeError('No data extracted — check §0 auth/project, §3 CORINE coverage, or lower R_LIST.')
scenes=pd.concat(frames, ignore_index=True).dropna(subset=['millis'])
scenes['date']=pd.to_datetime(scenes['millis'],unit='ms')
scenes['year']=scenes.date.dt.year; scenes['month']=scenes.date.dt.month
print('\nTotal rows:', len(scenes), '| arms:', sorted(scenes.arm.unique()), '| res:', sorted(scenes.res_m.unique()))

## 6 · Monthly composites (max NDVI, mean LST) + amplitude-by-resolution sanity check

In [ ]:
monthly=(scenes.groupby(['name','arm','res_m','year','month'])
         .agg(NDVI=('NDVI','max'),LST=('LST','mean'),n_scenes=('NDVI','size')).reset_index())

def amp(g):
    clim=g.groupby('month')['NDVI'].mean()
    return clim.max()-clim.min()
rows=[]
for (arm,R),g in monthly[monthly.name.isin(PLATEAU6)].groupby(['arm','res_m']):
    a=g.groupby('name').apply(amp).mean()
    rows.append({'arm':arm,'res_m':R,'plateau_NDVI_amplitude':round(float(a),3)})
print('Seasonal NDVI amplitude (plateau mean) — expect it to SHRINK as resolution coarsens, esp. unmasked (C):')
print(pd.DataFrame(rows).sort_values(['arm','res_m']).to_string(index=False))

## 7 · Export

In [ ]:
out='trakya_resolution_sensitivity.csv'
monthly.to_csv(out, index=False)
print('Wrote', out, '|', len(monthly), 'rows,', monthly.name.nunique(), 'districts,',
      monthly.arm.nunique(), 'arms')
try:
    from google.colab import files; files.download(out)
except Exception:
    print('Not in Colab — download', out, 'manually.')

## Notes
- Send `trakya_resolution_sensitivity.csv` back for the scale-selection (VHI–SPEI by scale) and coupling
  (VHI–SPEI-3, plateau vs İpsala) analysis by arm × resolution, and the figure.
- Timeout in §5? Set `START_YEAR = 2018`, or drop `900` from `R_LIST`.
- CORINE empty over Thrace? Swap the §3 mask for ESA WorldCover class 40 (see notebook 01).